In [5]:
import pandas as pd
from utils import drop_rows, drop_columns
import pyarrow.parquet as pq

In [6]:
file="../bronze/weather/weather.parquet"

In [7]:
df = pd.read_parquet(file)
#df
#df.info()

metadata = pq.read_metadata(file)
print(metadata.metadata)

{b'pandas': b'{"index_columns": [], "column_indexes": [], "columns": [{"name": "station", "field_name": "station", "pandas_type": "unicode", "numpy_type": "object", "metadata": null}, {"name": "date", "field_name": "date", "pandas_type": "unicode", "numpy_type": "object", "metadata": null}, {"name": "latitude", "field_name": "latitude", "pandas_type": "float64", "numpy_type": "float64", "metadata": null}, {"name": "longitude", "field_name": "longitude", "pandas_type": "float64", "numpy_type": "float64", "metadata": null}, {"name": "elevation", "field_name": "elevation", "pandas_type": "float64", "numpy_type": "float64", "metadata": null}, {"name": "name", "field_name": "name", "pandas_type": "unicode", "numpy_type": "object", "metadata": null}, {"name": "prcp", "field_name": "prcp", "pandas_type": "int64", "numpy_type": "int64", "metadata": null}, {"name": "prcp_attributes", "field_name": "prcp_attributes", "pandas_type": "unicode", "numpy_type": "object", "metadata": null}, {"name": "

DELETE ALL THE DATA BEFORE AND AFTER THE YEAR 2013

In [8]:
new_df= drop_rows(df,'date')
new_df
print(new_df.columns)
print(len(df))
print(len(new_df))

Index(['station', 'date', 'latitude', 'longitude', 'elevation', 'name', 'prcp',
       'prcp_attributes', 'snow', 'snow_attributes',
       ...
       'wt17', 'wt17_attributes', 'wt18', 'wt18_attributes', 'wt19',
       'wt19_attributes', 'wt21', 'wt21_attributes', 'wt22',
       'wt22_attributes'],
      dtype='object', length=124)
57606
365


INVESTIGATE WHICH TYPE OF DATA INCLUDES THE OBJECT TYPE

In [9]:
print(df.select_dtypes(include="object").dtypes)
for col in df.select_dtypes(include="object").columns:
    print(f"{col}: {df[col].dropna().head(1).tolist()}")

station            object
date               object
name               object
prcp_attributes    object
snow_attributes    object
                    ...  
wt17_attributes    object
wt18_attributes    object
wt19_attributes    object
wt21_attributes    object
wt22_attributes    object
Length: 62, dtype: object
station: ['USW00094728']
date: ['1869-01-01']
name: ['NY CITY CENTRAL PARK, NY US']
prcp_attributes: [',,Z,']
snow_attributes: [',,Z,']
snwd_attributes: [',,Z,']
tmax_attributes: [',,Z']
tmin_attributes: [',,Z']
acmh_attributes: [',,X']
acsh_attributes: [',,X']
adpt_attributes: [',,W']
aslp_attributes: [',,W']
astp_attributes: [',,W']
awbt_attributes: [',,W']
awnd_attributes: [',,X']
daev_attributes: [',,0']
dasf_attributes: [',,Z']
dawm_attributes: [',,0']
evap_attributes: [',,0']
fmtm_attributes: [',,X']
mdev_attributes: [',,0']
mdsf_attributes: [',,Z']
mdwm_attributes: [',,0']
pgtm_attributes: [',,X']
psun_attributes: [',,X']
rhav_attributes: [',,W']
rhmn_attributes: [',,W']
r

DELETE UNNECESSARY COLUMNS FOR THE ANALYSIS

In [10]:
index_delete_columns=new_df.columns[16:59]
name_delete_columns=['latitude','longitude','elevation']

first_delete_columns=drop_columns(new_df, index_delete_columns)
second_delete_columns=drop_columns(first_delete_columns, name_delete_columns)

second_delete_columns.head()

,station,date,name,prcp,prcp_attributes,snow,snow_attributes,snwd,snwd_attributes,tmax,...,wt17,wt17_attributes,wt18,wt18_attributes,wt19,wt19_attributes,wt21,wt21_attributes,wt22,wt22_attributes
56247,USW00094728,2023-01-01,"NY CITY CENTRAL PARK, NY US",0,"T,,W,2400",0.0,",,W,2400",0.0,",,W,2400",128.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56248,USW00094728,2023-01-02,"NY CITY CENTRAL PARK, NY US",5,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",133.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56249,USW00094728,2023-01-03,"NY CITY CENTRAL PARK, NY US",107,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",144.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56250,USW00094728,2023-01-04,"NY CITY CENTRAL PARK, NY US",5,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",189.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56251,USW00094728,2023-01-05,"NY CITY CENTRAL PARK, NY US",3,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",100.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None


GET ATTRIBUTES THAT INDICATED THE QUALITY OF INFORMATION

In [11]:
# Select all attribute columns
attribute_columns = [
    col for col in second_delete_columns.columns
    if col.endswith("_attributes")
]

# Extract quality flags from all attribute columns
quality_flags = second_delete_columns[attribute_columns].apply(
    lambda col: col.str.split(",").str[1]
)

# Identify non-empty quality flags
has_quality_issue = quality_flags.notna() & quality_flags.ne("")

# Keep rows with at least one quality issue
not_empty_data = second_delete_columns[
    has_quality_issue.any(axis=1)
]

print("Rows with quality issues:", len(not_empty_data))
quality_flags["prcp_attributes"].value_counts(dropna=False)
has_quality_issue.sum().sort_values(ascending=False)

Rows with quality issues: 0


prcp_attributes    0
snow_attributes    0
snwd_attributes    0
tmax_attributes    0
tmin_attributes    0
tobs_attributes    0
tsun_attributes    0
wdf1_attributes    0
wdf2_attributes    0
wdf5_attributes    0
wdfg_attributes    0
wdfm_attributes    0
wdmv_attributes    0
wesd_attributes    0
wsf1_attributes    0
wsf2_attributes    0
wsf5_attributes    0
wsfg_attributes    0
wsfm_attributes    0
wt01_attributes    0
wt02_attributes    0
wt03_attributes    0
wt04_attributes    0
wt05_attributes    0
wt06_attributes    0
wt07_attributes    0
wt08_attributes    0
wt09_attributes    0
wt11_attributes    0
wt13_attributes    0
wt14_attributes    0
wt15_attributes    0
wt16_attributes    0
wt17_attributes    0
wt18_attributes    0
wt19_attributes    0
wt21_attributes    0
wt22_attributes    0
dtype: int64

CHECK IF THE UNIT OF TEMPERATURES IS CELSIUS OR FAHRENHEIT

In [12]:
print(second_delete_columns[["tmax", "tmin", "snow", "snwd"]].describe())

             tmax        tmin        snow        snwd
count  365.000000  365.000000  365.000000  365.000000
mean   182.969863  106.008219    0.161644    0.219178
std     85.233078   77.922175    1.783212    3.048358
min    -27.000000 -160.000000    0.000000    0.000000
25%    117.000000   39.000000    0.000000    0.000000
50%    178.000000  100.000000    0.000000    0.000000
75%    256.000000  178.000000    0.000000    0.000000
max    339.000000  250.000000   23.000000   50.000000


TRANSFORM THE TEMPERATURES IN CELSIUS WITH DECIMAL

In [ ]:
second_delete_columns["tmax"]=second_delete_columns["tmax"]/10
second_delete_columns["tmin"]=second_delete_columns["tmin"]/10

second_delete_columns

,station,date,name,prcp,prcp_attributes,snow,snow_attributes,snwd,snwd_attributes,tmax,...,wt17,wt17_attributes,wt18,wt18_attributes,wt19,wt19_attributes,wt21,wt21_attributes,wt22,wt22_attributes
56247,USW00094728,2023-01-01,"NY CITY CENTRAL PARK, NY US",0,"T,,W,2400",0.0,",,W,2400",0.0,",,W,2400",12.8,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56248,USW00094728,2023-01-02,"NY CITY CENTRAL PARK, NY US",5,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",13.3,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56249,USW00094728,2023-01-03,"NY CITY CENTRAL PARK, NY US",107,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",14.4,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56250,USW00094728,2023-01-04,"NY CITY CENTRAL PARK, NY US",5,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",18.9,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56251,USW00094728,2023-01-05,"NY CITY CENTRAL PARK, NY US",3,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",10.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
56607,USW00094728,2023-12-27,"NY CITY CENTRAL PARK, NY US",81,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",10.0,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56608,USW00094728,2023-12-28,"NY CITY CENTRAL PARK, NY US",310,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",12.8,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56609,USW00094728,2023-12-29,"NY CITY CENTRAL PARK, NY US",13,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",12.8,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None
56610,USW00094728,2023-12-30,"NY CITY CENTRAL PARK, NY US",0,",,W,2400",0.0,",,W,2400",0.0,",,W,2400",7.8,...,NaN,None,NaN,None,NaN,None,NaN,None,NaN,None


CHECK IF EXISTS DUPLICATED BY METHEOROLOGICAL STATION AND IF THERE IS MISSING DATES

In [14]:
is_duplicated= second_delete_columns[second_delete_columns.duplicated(subset=["station","date"])]
print("Duplicated info: ",len(is_duplicated))
print(len(second_delete_columns))
print("First date:", second_delete_columns["date"].min())
print("Last date:", second_delete_columns["date"].max())

print("Unique dates:", second_delete_columns["date"].nunique())
print("Unique stations:", second_delete_columns["station"].nunique())

Duplicated info:  0
365
First date: 2023-01-01
Last date: 2023-12-31
Unique dates: 365
Unique stations: 1
